# Customer 360 View
## Project Execution Guidelines

### Step 1: Data Loading and Initial Exploration

* Load all datasets using Pandas
* Inspect structure using `.head()`, `.info()`, `.describe()`
* Identify primary and foreign keys
* Understand relationships between tables

In [11]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import glob
#!pip install pathlib
from pathlib import Path
current_dir = Path.cwd()
data_folder = current_dir/"python_project_aiml_logicmojo_dataset"
#print(data_folder)
data_files = glob.glob(f"{data_folder}/*.csv")
dataframes = {}
#Load all datasets using Pandas
for file_path_str in data_files:
    file_path = Path(file_path_str)
    #print(file_path)
    key = file_path.stem
    try:
        dataframes[key] = pd.read_csv(file_path)
        # Inspect structure using
        print(f"\nFirst {key} 3 rows:")
        print(dataframes[key].head(3))
        print(f"\nInformation about {key}:")
        print(dataframes[key].info())
        print(f"\nInformation about {key} Shape:")
        print(dataframes[key].shape)
        print(f"\n {key} Summary Statistics:")
        print(dataframes[key].describe())
        print(f"\n  {key} Missing Values:")
        print(dataframes[key].isnull().sum())
        print(f"\n  {key} Duplicate Values:")
        print(dataframes[key].duplicated().sum())
        
    except Exception as e:
        print(f"Failed to load {var_name} : {e}")


First customers 3 rows:
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  

Information about customers:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3

In [12]:
all_column_lists = []
for df in dataframes.values():
    all_column_lists.append(list(df.columns))
#print(all_column_lists)
all_column_sets = []
for col_list in all_column_lists:
    all_column_sets.append(set(col_list))
print(all_column_sets)

[{'customer_state', 'customer_id', 'customer_unique_id', 'customer_city', 'customer_zip_code_prefix'}, {'product_name_lenght', 'product_length_cm', 'product_description_lenght', 'product_id', 'product_weight_g', 'product_category_name', 'product_height_cm', 'product_photos_qty', 'product_width_cm'}, {'order_id', 'review_score', 'review_id', 'review_comment_message', 'review_creation_date', 'review_comment_title', 'review_answer_timestamp'}, {'order_id', 'order_approved_at', 'customer_id', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_purchase_timestamp', 'order_delivered_carrier_date', 'order_status'}, {'product_category_name', 'product_category_name_english'}, {'geolocation_city', 'geolocation_lat', 'geolocation_state', 'geolocation_lng', 'geolocation_zip_code_prefix'}, {'order_id', 'payment_type', 'payment_installments', 'payment_sequential', 'payment_value'}, {'seller_city', 'seller_id', 'seller_state', 'seller_zip_code_prefix'}, {'order_id', 'freight_valu

In [13]:
# Manually assigning each table to its own explicit variable
customers            = dataframes['customers'].copy()
products             = dataframes['products'].copy()
reviews              = dataframes['reviews'].copy()
orders               = dataframes['orders'].copy()
category_translation = dataframes['category_translation'].copy()
location             = dataframes['location'].copy()
payments             = dataframes['payments'].copy()
sellers              = dataframes['sellers'].copy()
order_item           = dataframes['order_item'].copy()


### Visualising the Data

In [14]:
# fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# # Left plot: Products nulls
# sns.heatmap(products.isnull(), cbar=False, yticklabels=False, cmap='viridis', ax=axes[0])
# axes[0].set_title("Missing Data: products")

# # Right plot: Orders nulls
# sns.heatmap(orders.isnull(), cbar=False, yticklabels=False, cmap='plasma', ax=axes[1])
# axes[1].set_title("Missing Data: orders")

# plt.tight_layout()
# plt.show()

In [15]:
# plt.figure(figsize=(12, 5))

# # Plotting item prices under $300 to cut out long-tail outliers for cleaner visualization
# sns.kdeplot(data=order_item[order_item['price'] < 300], x='price', fill=True, label='Item Unit Price', color='teal')
# sns.kdeplot(data=payments[payments['payment_value'] < 300], x='payment_value', fill=True, label='Total Payment Value', color='coral')

# plt.title("Distribution Density: Unit Prices vs Total Payment Values (< $300)")
# plt.xlabel("Value (Currency Units)")
# plt.ylabel("Density")
# plt.legend()
# plt.show()

In [16]:
# plt.figure(figsize=(10, 5))

# # Order states from highest volume to lowest
# state_order = customers['customer_state'].value_counts().index

# # Using a standard, built-in Seaborn palette name ('Blues_r' for reversed blue gradient)
# sns.countplot(data=customers, x='customer_state', order=state_order, palette='Blues_r')

# plt.title("Volume of Unique Customers Across States")
# plt.xlabel("State Code")
# plt.ylabel("Customer Count")
# plt.show()

### Step 2: Data Cleaning and Preprocessing

* Handle missing values appropriately
* Remove duplicate records
* Convert date columns to datetime format
* Validate data types and ranges
* Standardize column names if required

In [17]:
df_to_clean = {
    'customers': customers, 'products': products, 'reviews': reviews, 
    'orders': orders, 'category_translation': category_translation, 
    'location': location, 'payments': payments, 'sellers': sellers, 
    'order_item': order_item
}
#Making sure column names are consistent
for name, df in df_to_clean.items():
    df.columns.str.strip().str.lower().str.replace(' ', '_')

In [18]:
#Remove duplicate records
for name, df in df_to_clean.items():
    initial_shape = df.shape
    dupli = df.duplicated().sum()
    if dupli > 0:
        df.drop_duplicates(inplace=True)
        print(f"{name}: Removed {dupli} duplicates. Old Shape {initial_shape}, new shape {df.shape}")
    else:
        print(f"{name}: Zero duplicates found. Shape {df.shape}")


customers: Zero duplicates found. Shape (99441, 5)
products: Zero duplicates found. Shape (32951, 9)
reviews: Zero duplicates found. Shape (99224, 7)
orders: Zero duplicates found. Shape (99441, 8)
category_translation: Zero duplicates found. Shape (71, 2)
location: Removed 261831 duplicates. Old Shape (1000163, 5), new shape (738332, 5)
payments: Zero duplicates found. Shape (103886, 5)
sellers: Zero duplicates found. Shape (3095, 4)
order_item: Zero duplicates found. Shape (112650, 7)


In [19]:
#Fix the date format
#Reviews Table
df_to_clean['reviews']['review_creation_date'] = pd.to_datetime(df_to_clean['reviews']['review_creation_date'],errors='coerce')
df_to_clean['reviews']['review_answer_timestamp'] = pd.to_datetime(df_to_clean['reviews']['review_creation_date'],errors='coerce')
#Order Table
df_to_clean['orders']['order_purchase_timestamp'] = pd.to_datetime(df_to_clean['reviews']['review_creation_date'],errors='coerce')
df_to_clean['orders']['order_approved_at'] = pd.to_datetime(df_to_clean['reviews']['review_creation_date'],errors='coerce')
df_to_clean['orders']['order_delivered_carrier_date'] = pd.to_datetime(df_to_clean['reviews']['review_creation_date'],errors='coerce')
df_to_clean['orders']['order_delivered_customer_date'] = pd.to_datetime(df_to_clean['reviews']['review_creation_date'],errors='coerce')
df_to_clean['orders']['order_estimated_delivery_date'] = pd.to_datetime(df_to_clean['reviews']['review_creation_date'],errors='coerce')
#Order Item Table
order_item['shipping_limit_date'] = pd.to_datetime(dataframes['order_item']['shipping_limit_date'], format='mixed', errors='coerce')

Missing values - 
Random Missing - Manageable
Pattern Suspecious
Few rows - Drop them
Missing column - fill it up
Text comment keep NAN
Critical numeric - impute some values.
Drop - Drop NA
Fill - Fill NA
Isnull - check missing data.
Duplicate Records - Show detection - Show fixing step - Verification step
(add inline comments)
Date converstion - To Datetime - Show with info (Verify)
Datatype - numberic, string, object - Correct data type- .info 
Column Standardisation- Lower case with underscore
data quality report

In [20]:
# Products - remove 2 rows with missing dimensions
products.dropna(
    subset=[
        'product_weight_g',
        'product_length_cm',
        'product_height_cm',
        'product_width_cm'
    ],
    inplace=True
)

# Fill product metadata
products['product_category_name'] = products['product_category_name'].fillna('unknown')
products['product_name_lenght'] = products['product_name_lenght'].fillna(0)
products['product_description_lenght'] = products['product_description_lenght'].fillna(0)
products['product_photos_qty'] = products['product_photos_qty'].fillna(0)

# Reviews
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('No Title')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('No Message')

# Payments
payments = payments[
    (payments['payment_value'] >= 0) &
    (payments['payment_installments'] >= 0)
]

# Order Items
order_item = order_item[
    (order_item['price'] > 0) &
    (order_item['freight_value'] >= 0)
]

# Product dimensions validation
products = products[
    (products['product_weight_g'] > 0) &
    (products['product_length_cm'] > 0) &
    (products['product_height_cm'] > 0) &
    (products['product_width_cm'] > 0)
]
delivered_orders = df_to_clean['orders'][df_to_clean['orders']['order_status'] == 'delivered'].copy()

In [21]:
for name, df in df_to_clean.items():
    print(f"\n{name}")
    print(df.isnull().sum().sum(), "missing values")
    print(df.duplicated().sum(), "duplicates")
    print(df.dtypes)


customers
0 missing values
0 duplicates
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

products
0 missing values
0 duplicates
product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

reviews
0 missing values
0 duplicates
review_id                          object
order_id                           object
review_score                        int64
review_comment_title               object
review_comment_message             object
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object

orders
1085 missing values

In [22]:
# #Drop minor missing data only 2
# products.dropna(subset=[
# 'product_weight_g',
# 'product_length_cm',
# 'product_height_cm',
# 'product_width_cm'
# ], inplace=True)

# products['product_category_name'] = (
#     products['product_category_name']
#     .fillna('unknown')
# )
# products['product_name_lenght'] = (
#     products['product_name_lenght']
#     .fillna('0')
# )
# products['product_description_lenght'] = (
#     products['product_description_lenght']
#     .fillna('0')
# )
# products['product_photos_qty'] = (
#     products['product_photos_qty']
#     .fillna('0')
# )
# #Verify
# #print(products[['product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty']].info())

# # Missing reviews data
# reviews['review_comment_title'] = reviews['review_comment_title'].fillna('No Title')
# reviews['review_comment_message'] = reviews['review_comment_message'].fillna('No Message')

# #Verify
# #print(reviews.isnull().sum())

# # Rows with zero or less:
# payments = payments[(payments['payment_value'] >= 0) & (payments['payment_installments'] >= 0)]
# order_item = order_item[(order_item['price'] > 0) & (order_item['freight_value'] >= 0)]
# products = products[
#     (products['product_weight_g'] > 0) & 
#     (products['product_length_cm'] > 0) & 
#     (products['product_height_cm'] > 0) & 
#     (products['product_width_cm'] > 0)
# ]

# products = products.dropna(subset=['product_category_name', 'product_weight_g'])
# order_item = order_item[(order_item['price'] > 0) & (order_item['freight_value'] >= 0)]
# Keep valid missing dates
# orders['order_approved_at'] = orders['order_approved_at'].fillna(pd.NaT)
# orders['order_delivered_carrier_date'] = orders['order_delivered_carrier_date'].fillna(pd.NaT)
# orders['order_delivered_customer_date'] = orders['order_delivered_customer_date'].fillna(pd.NaT)

In [23]:
summary_data = []
for name, df in df_to_clean.items():
    summary_data.append({
        'Dataset': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Remaining Missing': df.isnull().sum().sum(),
        'Remaining Duplicates': df.duplicated().sum()
    })

quality_report_df = pd.DataFrame(summary_data)
print(quality_report_df.to_string(index=False))

             Dataset   Rows  Columns  Remaining Missing  Remaining Duplicates
           customers  99441        5                  0                     0
            products  32949        9                  0                     0
             reviews  99224        7                  0                     0
              orders  99441        8               1085                     0
category_translation     71        2                  0                     0
            location 738332        5                  0                     0
            payments 103886        5                  0                     0
             sellers   3095        4                  0                     0
          order_item 112650        7                  0                     0


### Step 3: Data Integration (Critical Component)

You must construct a **Master Dataset** by merging multiple tables.

Recommended sequence:

1. orders + customers
2. orders + order_items
3. order_items + products
4. orders + payments
5. orders + reviews
6. order_items + sellers
7. products + category_translation

**Final output:** A **single consolidated dataset** representing a unified business view

In [24]:
#1. orders + customers
orders_customers = pd.merge(orders, customers, on='customer_id', how='left')
#2. orders + order_items
orders_items = pd.merge(order_item, orders_customers, on='order_id', how='left')
#3. order_items + products
items_products = pd.merge(orders_items, products, on='product_id', how='left')
#4. orders + payments
items_payments = pd.merge(items_products, payments, on='order_id', how='left')
#5. orders + reviews
items_reviews = pd.merge(items_payments, reviews, on='order_id', how='left') 
#6. order_items + sellers
items_sellers = pd.merge(items_reviews, sellers, on='seller_id', how='left')
#7. products + category_translation
master_df = pd.merge(items_sellers, category_translation, on='product_category_name', how='left')

In [25]:
print(master_df.head())

                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   
3  00024acbcdf0a6daa1e931b038114c75              1   
4  00042b26cf59d7ce69dfabb4e55b4fd9              1   

                         product_id                         seller_id  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36   
2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d   
3  7634da152a4610f1595efa32f14722fc  9d7a1d34a5052409006425275ba1c2b4   
4  ac6c3623068f30de03045865e4e10089  df560393f3a51e74553ab94004ba5c87   

  shipping_limit_date   price  freight_value  \
0 2017-09-19 09:45:35   58.90          13.29   
1 2017-05-03 11:05:13  239.90          19.93   
2 2018-01-18 14:48:30  199.00          17.87   
3 2018-08-15 10:10:18   12.99          12.79

Data integration - Building customer 360

Customer system
Order system
product system
Payment system
Review System

One review everything about customer + order + product + payments + reviews

In [ ]:
customer_360 = master_df.groupby('customer_id').agg(
    #customer info
    customer_unique_id=('customer_unique_id', 'first'),
    customer_city=('customer_city','first'),
    customer_state=('customer_state', 'first'),
    #Order System
    total_orders=('order_id', 'nunique'),
    total_items_bought=('product_id', 'count'),
    first_purchase_date=('order_purchase_timestamp', 'min'),
    latest_purchase_date=('order_purchase_timestamp', 'max'),
    order_statuses=('order_status', lambda x: list(x.unique()))
    
)

### Step 4: Feature Engineering

Create meaningful features such as:

* Total order value (aggregated from order_items or payments)
* Delivery time (order purchase to delivery date)
* Number of items per order
* Customer purchase frequency
* Customer lifetime value (basic approximation)
* Average order value per customer

### Step 5: Exploratory Data Analysis (EDA)

Perform structured analysis across the following dimensions:

#### Customer Analysis

* New vs repeat customers
* High-value vs low-value customers
* Geographic distribution of customers

#### Revenue and Order Analysis

* Monthly revenue trends
* Order volume trends
* Peak sales periods

#### Product Analysis

* Top-selling product categories
* Revenue contribution by category
* Product demand distribution

#### Seller Analysis

* Top-performing sellers
* Seller contribution to revenue
* Seller distribution

#### Review and Satisfaction Analysis

* Distribution of review scores
* Relationship between delivery time and ratings
* Identification of dissatisfaction patterns

### Step 6: Data Visualization

Use Matplotlib and Seaborn to create:

* Time series plots (sales trends)
* Bar charts (category performance)
* Histograms (distribution analysis)
* Box plots (outlier detection)
* Heatmaps (correlation analysis)

All visualizations must be clearly labeled and interpretable.

### Step 7: Business Insights and Recommendations

You must derive clear and actionable insights:

* Identify top revenue-driving factors
* Highlight customer behavior patterns
* Evaluate operational inefficiencies
* Provide strategic recommendations

Insights must be supported by data and visual evidence.